# Phase 7 — Secure evidence retrieval and reference recommendations

## Goal

Produce a reproducible, evidence-backed shortlist from one approved Phase 6
opportunity. This fast baseline uses the signed BM25 index: no new model download,
no external LLM, no cross-encoder, and no source-corpus mutation.

The packaged redacted sample validates engineering only. A real shortlist remains
subject to business review and the independent Phase 5.1 expert evaluation gate.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

PROJECT_FOLDER_NAME = "Devoteam_AI_CLEAN_PIPELINE"
PROJECT_PARENT_FOLDER_NAME = "Devoteam internship"
PACKAGE_FILENAME = "PHASE_7_EVIDENCE_RECOMMENDATIONS_PACKAGE.zip"
PACKAGE_SHA256 = "fa3b62f3e3b83f8b79b6551c104abbd4cc6708692193c9de6ec89f089fc38e3d"
PACKAGE_MANIFEST_SHA256 = "b78f47c89b5255dc410aa2eb899c737f92784db85000eafa06d09f7456ccc41d"
SNAPSHOT_ID = "20260714T154731Z_129ff982c8"
PHASE4_RUN_NAME = "phase4_corpus_v1"
PHASE5_RUN_NAME = "phase5_hybrid_retrieval_v1"

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("Phase 7 contract loaded: secure BM25 baseline; external model calls disabled.")

## 1. Locate the clean project

The Colab path is explicit so Drive discovery cannot stall. Local validation may
set `DEVOTEAM_PROJECT_ROOT` and `DEVOTEAM_PHASE7_PACKAGE`.

In [ ]:
override = os.environ.get("DEVOTEAM_PROJECT_ROOT")
if override:
    PROJECT_ROOT = Path(override).resolve()
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = (
        Path("/content/drive/MyDrive")
        / PROJECT_PARENT_FOLDER_NAME
        / PROJECT_FOLDER_NAME
    ).resolve()
    assert PROJECT_ROOT.is_dir(), f"Clean project folder not found: {PROJECT_ROOT}"
    assert (PROJECT_ROOT / "config" / "project.yaml").exists(), "Project configuration is missing"

PACKAGE_PATH = Path(
    os.environ.get("DEVOTEAM_PHASE7_PACKAGE", PROJECT_ROOT / PACKAGE_FILENAME)
).resolve()
assert PROJECT_ROOT.name == PROJECT_FOLDER_NAME, PROJECT_ROOT
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
print(f"Project root: {PROJECT_ROOT}")

## 2. Verify and install the signed Phase 7 overlay

Only manifest-listed files are installed. Identical files are skipped and any
conflict stops the run. The default test is Phase 7 only for speed; set
`DEVOTEAM_RUN_FULL_REGRESSION=1` to rerun every project test.

In [ ]:
assert file_sha256(PACKAGE_PATH) == PACKAGE_SHA256, "Phase 7 package hash mismatch"
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    names = archive.namelist()
    assert "PHASE_7_PACKAGE_MANIFEST.json" in names
    manifest_bytes = archive.read("PHASE_7_PACKAGE_MANIFEST.json")
    assert hashlib.sha256(manifest_bytes).hexdigest() == PACKAGE_MANIFEST_SHA256
    package_manifest = json.loads(manifest_bytes)
    allowed = set(package_manifest["files"]) | {"PHASE_7_PACKAGE_MANIFEST.json"}
    assert set(names) == allowed, "Package contains undeclared files"
    installed = skipped = 0
    for name in names:
        target = (PROJECT_ROOT / name).resolve()
        assert target == PROJECT_ROOT or PROJECT_ROOT in target.parents, name
        data = archive.read(name)
        if name != "PHASE_7_PACKAGE_MANIFEST.json":
            assert hashlib.sha256(data).hexdigest() == package_manifest["files"][name]["sha256"]
        if target.exists():
            assert target.read_bytes() == data, f"Conflicting existing Phase 7 file: {name}"
            skipped += 1
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            installed += 1

requirements = PROJECT_ROOT / "requirements" / "phase7.txt"
if os.environ.get("DEVOTEAM_SKIP_PIP") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
test_target = PROJECT_ROOT / ("tests" if os.environ.get("DEVOTEAM_RUN_FULL_REGRESSION") == "1" else "tests/test_phase7_recommendations.py")
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_target)],
    cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True,
)
print(tests.stdout[-4000:])
assert tests.returncode == 0, tests.stderr[-4000:]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Package verified: installed={installed}, unchanged={skipped}")

## 3. Resolve and verify the signed inputs

Phase 7 consumes the immutable Phase 4 catalogues, signed Phase 5 BM25 files,
and exactly one complete Phase 6 run. Set `DEVOTEAM_PHASE6_OPPORTUNITY_ID` only
when several opportunity folders exist.

In [ ]:
from devoteam_reference_ai.phase5_retrieval import load_phase5_config
from devoteam_reference_ai.phase6_opportunity import load_phase6_config
from devoteam_reference_ai.phase7_recommendations import (
    load_phase7_config,
    run_phase7,
    verify_phase7_run,
)

PHASE5_CONFIG = load_phase5_config(PROJECT_ROOT / "config" / "phase5_retrieval.yaml")
PHASE6_CONFIG = load_phase6_config(PROJECT_ROOT / "config" / "phase6_opportunity.yaml")
PHASE7_CONFIG = load_phase7_config(PROJECT_ROOT / "config" / "phase7_recommendations.yaml")
PHASE4_ROOT = PROJECT_ROOT / "data" / "canonical" / SNAPSHOT_ID / PHASE4_RUN_NAME
PHASE5_ROOT = PROJECT_ROOT / "data" / "indexes" / SNAPSHOT_ID / PHASE5_RUN_NAME

opportunity_id = os.environ.get("DEVOTEAM_PHASE6_OPPORTUNITY_ID", "").strip()
if opportunity_id:
    PHASE6_ROOT = PROJECT_ROOT / "data" / "opportunities" / opportunity_id / PHASE6_CONFIG["pipeline_version"]
    assert PHASE6_ROOT.is_dir(), PHASE6_ROOT
else:
    candidates = sorted(
        path for path in (PROJECT_ROOT / "data" / "opportunities").glob(f"*/{PHASE6_CONFIG['pipeline_version']}")
        if (path / PHASE6_CONFIG["output"]["success_marker"]).exists()
    )
    assert len(candidates) == 1, (
        "Expected one complete Phase 6 run; set DEVOTEAM_PHASE6_OPPORTUNITY_ID. "
        f"Found: {candidates}"
    )
    PHASE6_ROOT = candidates[0]

print(f"Phase 4: {PHASE4_ROOT}")
print(f"Phase 5: {PHASE5_ROOT}")
print(f"Phase 6: {PHASE6_ROOT}")

## 4. Retrieve, score, validate, and sign

Security and approved hard filters are applied before BM25 scoring. Compatible
filters are reapplied to canonical reference metadata after document mapping to
prevent cross-reference leakage.

In [ ]:
RUN_ROOT, MANIFEST = run_phase7(
    phase5_root=PHASE5_ROOT,
    phase4_root=PHASE4_ROOT,
    phase6_root=PHASE6_ROOT,
    phase5_config=PHASE5_CONFIG,
    phase6_config=PHASE6_CONFIG,
    phase7_config=PHASE7_CONFIG,
    output_root=PROJECT_ROOT / PHASE7_CONFIG["output"]["root"],
)
verification = verify_phase7_run(RUN_ROOT, PHASE7_CONFIG)
assert verification["manifest"] == MANIFEST

print("PHASE 7 EVIDENCE RECOMMENDATIONS: PASS")
print(f"Status: {MANIFEST['status']}")
print(f"Recommendations: {MANIFEST['recommendations']}")
print(f"Evidence rows: {MANIFEST['evidence_rows']}")
print(f"Requirement gaps: {MANIFEST['requirement_gaps']}")
print(f"Citation coverage: {MANIFEST['citation_coverage']:.0%}")
print("External LLM / embedding / cross-encoder calls: 0 / 0 / 0")
print(f"Output: {RUN_ROOT}")
if MANIFEST["status"] == "TECHNICAL_PASS_SAMPLE_ONLY":
    print("IMPORTANT: sample validation only; do not use this shortlist as a client deliverable.")

## Next step

Open `RECOMMENDATION_REVIEW.xlsx` in the printed output folder. For a real
opportunity, a business reviewer marks candidate rows `SHORTLIST` or `REJECT`
and records notes. Template generation must consume only an approved shortlist.
The independent Phase 5.1 expert evaluation remains required before production
promotion.